In [17]:
import os
import openai
import constantes as c
import callingLLMs as llm
import annotation as ann
import annotation_v2 as ann2

In [6]:
# Chemins
path_to_api_key = "/Users/charles-/Downloads/clefs/OPENROUTER_KEY_PERSO.txt"
path_to_res_folder = "./results_v2"

## Calling LLMs

In [7]:
with open(path_to_api_key, "r") as file:
    API_KEY = file.read().strip()
client = llm.get_client(API_KEY)

In [8]:
os.makedirs(path_to_res_folder, exist_ok=True)
test = llm.run_all(prompt=c.PROMPT2, models_by_category={"test": ["google/gemma-4-31b-it"]},
                   n_calls=1, client=client, save_path=f"{path_to_res_folder}/test.jsonl")
print(test["answer"][0])

0 appel(s) déjà faits, 1 à effectuer (sauvegarde : ./results_v2/test.jsonl)
{DOMAINE} : Administration funéraire

R : Le remboursement des frais d'obsèques est conditionné à la présentation d'un certificat de décès original et non retouché.

N : On s'attendrait à ce que la preuve du décès soit suffisante pour déclencher le paiement, indépendamment de l'état physique du document.

Énoncé : La veuve a fait tomber le certificat de décès dans la soupe à l'oignon. Elle a tenté de repasser le papier mouillé pour en effacer les taches de bouillon. Le comptable de l'assurance a refusé le dossier en pointant la texture gondolée du document. La veuve a alors replacé le corps du défunt dans le cercueil et l'a ramené au salon pour y habiter.


In [9]:
os.makedirs(path_to_res_folder, exist_ok=True)
df = llm.run_all(prompt=c.PROMPT2, models_by_category=c.MODELES, client=client, n_calls=3,
                 save_path=f"{path_to_res_folder}/generations.jsonl", max_workers=4)
df.to_csv(f"{path_to_res_folder}/generations.csv", index=False)

1 appel(s) déjà faits, 74 à effectuer (sauvegarde : ./results_v2/generations.jsonl)


In [10]:
erreurs = df[df["answer"].str.startswith("ERROR")]
print(f"{len(df)} réponses, dont {len(erreurs)} erreurs")
print(erreurs.groupby("model")["answer"].first().str[:150].to_string())

75 réponses, dont 0 erreurs
Series([], )


In [11]:
print(erreurs.groupby("model")["answer"].first().to_string())

ids = sorted(m.id for m in client.models.list())
for motif in ["deepseek-r1", "lfm", "qwq"]:
    print(motif, "→", [i for i in ids if motif in i])
    

Series([], )
deepseek-r1 → ['deepseek/deepseek-r1-0528']
lfm → ['liquid/lfm-2.5-24b-a2b', 'liquid/lfm-3-1.7b', 'liquid/lfm-3-1.7b:free']
qwq → []


In [12]:
for m in client.models.list():
    if "lfm" in m.id:
        extra = m.model_extra or {}
        raisonne = "reasoning" in (extra.get("supported_parameters") or [])
        print(m.id, "| raisonne :", raisonne, "|", (extra.get("description") or "")[:150])

liquid/lfm-3-1.7b | raisonne : True | A compact 1.7B-parameter text model with a 256K-token context, post-trained for agents, tool calling, and structured outputs. It reasons before it ans
liquid/lfm-3-1.7b:free | raisonne : True | A compact 1.7B-parameter text model with a 256K-token context, post-trained for agents, tool calling, and structured outputs. It reasons before it ans
liquid/lfm-2.5-24b-a2b | raisonne : True | LFM2.5-24B-A2B is a mixture-of-experts reasoning model from Liquid AI with 24B total and 2.45B active parameters. It always reasons before answering a


## Synthesizing into table

In [18]:
# Un fichier Excel par annotateur : ordre mélangé, modèles masqués, critères en 0/1
ann2.exporter_annotation(df, annotateurs=("Casanove", "Charles"), dossier=path_to_res_folder)

[PosixPath('results_v2/annotation_Casanove.xlsx'),
 PosixPath('results_v2/annotation_Charles.xlsx')]

In [19]:
# Après annotation (fichiers enregistrés dans Excel ou LibreOffice) : accord critère par critère
ann2.comparer_annotations(f"{path_to_res_folder}/annotation_Casanove.xlsx",
                         f"{path_to_res_folder}/annotation_Charles.xlsx")

,critère,kappa,n
0,E Effet,None,0
1,1a Cadre nommable,None,0
2,"1b Non résolu, non commenté",None,0
3,1c Monde ordinaire,None,0
4,2a Règle,None,0
5,2b Consistance,None,0
6,2c Enjeu et décalage,None,0
7,3a Rigueur,None,0
8,3b Sans intérêt local,None,0
9,3c Productivité,None,0
